In [2]:
import os
import subprocess
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

# Load your database
df = pd.read_csv("../data/DB for chromophore_Sci_Data_rev02.csv")

In [ ]:
# Optical Screening 
df['extinction_coef'] = 10 ** df['log(e/mol-1 dm3 cm-1)']
df['LHE'] = 1 - (10 ** -df['extinction_coef'])
# Filter for promising optical properties
good_absorbers = df[(df['Absorption max (nm)'] > 450) & (df['log(e/mol-1 dm3 cm-1)'] > 4.0)]

#  Structural Screening with RDKit SMARTS 
# Define common anchoring groups
anchors = {
    "carboxylic_acid": Chem.MolFromSmarts("C(=O)[O;H1,B]"),
    "cyanoacrylate": Chem.MolFromSmarts("C(#N)=C-C(=O)O"),
    "phosphonic_acid": Chem.MolFromSmarts("P(=O)(O)O")
}

def check_anchors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return "Invalid"
    for name, pattern in anchors.items():
        if mol.HasSubstructMatch(pattern):
            return name
    return "None"

df['Anchoring_Group'] = df[df.columns[1]].apply(check_anchors)

#  xTB Energy Calculations 
def run_xtb_analysis(smiles, mol_id):
    # Convert SMILES to 3D Mol block
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    AllChem.MMFFOptimizeMolecule(mol)
    
    # Write to an XYZ file for xTB
    xyz_file = f"mol_{mol_id}.xyz"
    Chem.MolToXYZFile(mol, xyz_file)
    
    # Run GFN2-xTB geometry optimization
    # --opt performs structural relaxation, outputting orbital energies
    cmd = f"xtb {xyz_file} --opt --gfn 2"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
   
    # Parse xTB output for HOMO and LUMO (simplified regex/string search logic)
    homo, lumo = None, None
    if os.path.exists("xtbresults.pkt") or "Orbital Energies" in result.stdout:
        with open("xtbout.txt", "w") as f: f.write(result.stdout)
        # Custom parsing of 'xtbout.txt' to find the HOMO/LUMO gap line
        # Extract values usually listed under "(ev)" columns
        pass 
        
    # Clean up files
    # os.remove(xyz_file)
    return homo, lumo


In [ ]:
# run for the first few rows
for idx, row in good_absorbers.head(5).iterrows():
    if row['Anchoring_Group'] != "None":
        print(f"Running xTB for Molecule {idx} with anchor {row['Anchoring_Group']}...")
        homo, lumo = run_xtb_analysis(row['SMILE'], idx)

0        False
1        False
2        False
3        False
4        False
         ...  
20231    False
20232    False
20233    False
20234    False
20235    False
Name: Anchoring_Group, Length: 20236, dtype: bool